In [1]:
import pandas as pd

print("Pandas imported successfully!")

Pandas imported successfully!


In [2]:
train_df = pd.read_csv(
    "../data/raw/tweet_eval_sentiment_train.csv"
)

print("Dataset loaded successfully!")
print("Shape:", train_df.shape)

Dataset loaded successfully!
Shape: (45615, 4)


In [3]:
print(train_df.columns.tolist())

['text', 'label', 'text_length', 'sentiment']


In [4]:
df = train_df[["text", "sentiment"]].copy()

print(df.head())
print("Shape:", df.shape)

                                                text sentiment
0  "QT @user In the original draft of the 7th boo...  Positive
1  "Ben Smith / Smith (concussion) remains out of...   Neutral
2  Sorry bout the stream last night I crashed out...   Neutral
3  Chase Headley's RBI double in the 8th inning o...   Neutral
4  @user Alciato: Bee will invest 150 million in ...  Positive
Shape: (45615, 2)


In [5]:
print("Missing values before cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
text         0
sentiment    0
dtype: int64


In [6]:
df = df.dropna(subset=["text", "sentiment"]).copy()

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
text         0
sentiment    0
dtype: int64


In [7]:
df["text"] = df["text"].astype(str).str.strip()

df = df[df["text"] != ""].copy()

print("Dataset shape:", df.shape)

Dataset shape: (45615, 2)


In [8]:
print("Duplicates before:",
      df["text"].duplicated().sum())

Duplicates before: 29


In [9]:
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Duplicates after:",
      df["text"].duplicated().sum())

print("New shape:", df.shape)

Duplicates after: 0
New shape: (45586, 2)


In [10]:
print(df["sentiment"].value_counts())

sentiment
Neutral     20655
Positive    17840
Negative     7091
Name: count, dtype: int64


In [11]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

print("VADER initialized successfully!")

VADER initialized successfully!


In [12]:
text = "The camera is excellent but the battery is terrible."

scores = analyzer.polarity_scores(text)

print(scores)

{'neg': 0.307, 'neu': 0.519, 'pos': 0.174, 'compound': -0.4215}


In [13]:
def is_ambivalent_candidate(text):
    scores = analyzer.polarity_scores(text)

    positive = scores["pos"]
    negative = scores["neg"]

    return positive >= 0.20 and negative >= 0.20

In [14]:
test_texts = [
    "The camera is excellent but the battery is terrible.",
    "I absolutely love this product.",
    "This product is horrible."
]

for text in test_texts:
    print("Text:", text)
    print("Candidate:", is_ambivalent_candidate(text))
    print("-" * 60)

Text: The camera is excellent but the battery is terrible.
Candidate: False
------------------------------------------------------------
Text: I absolutely love this product.
Candidate: False
------------------------------------------------------------
Text: This product is horrible.
Candidate: False
------------------------------------------------------------


In [ ]:
df["ambivalent_candidate"] = df["text"].apply(
    is_ambivalent_candidate
)

print(df["ambivalent_candidate"].value_counts())

In [ ]:
candidates = df[
    df["ambivalent_candidate"] == True
].copy()

print("Number of candidates:", len(candidates))

Number of candidates: 239


In [ ]:
for i, text in enumerate(candidates["text"].head(20)):
    print(f"{i+1}. {text}")
    print("-" * 80)

1. Yeah well done dickhead so the IRA and KKK are Muslim #narrowmindedprick
--------------------------------------------------------------------------------
2. Grateful Dead plays The Beatles   Though it struck me that the latest gens may not know...
--------------------------------------------------------------------------------
3. @user @user Bloody Sunday was terrible. And yes the IRA killed 11 innocent musicians. I forgive. But I want justice.
--------------------------------------------------------------------------------
4. so happy tomorrow is Star Wars day!!
--------------------------------------------------------------------------------
5. .@Microsoft's patch Tuesday incorporated 56 fixes for #security #vulnerabilities - some with active #exploits insane!
--------------------------------------------------------------------------------
6. Wade be playing like Tracy Mcgrady in the 1st half Lol just losing the ball an shit
---------------------------------------------------------

In [ ]:
candidates[["text"]].to_csv(
    "../data/raw/ambivalent_candidates.csv",
    index=False
)

print("Candidate file saved successfully!")

Candidate file saved successfully!


In [1]:
import pandas as pd

ambivalent_df = pd.read_csv(
    "../data/raw/ambivalent_candidates.csv"
)

print("Ambivalent candidate file loaded!")
print("Shape:", ambivalent_df.shape)
print("Columns:", ambivalent_df.columns.tolist())

Ambivalent candidate file loaded!
Shape: (239, 2)
Columns: ['text', 'ambivalent_label']


In [2]:
print(ambivalent_df["ambivalent_label"].value_counts())

ambivalent_label
0    203
1     36
Name: count, dtype: int64


In [3]:
ambivalent_df = ambivalent_df[
    ambivalent_df["ambivalent_label"] == 1
].copy()

print("Verified Ambivalent samples:", len(ambivalent_df))

Verified Ambivalent samples: 36


In [4]:
ambivalent_df["sentiment"] = "Ambivalent"

print(
    ambivalent_df[["text", "sentiment"]].head(10)
)

                                                 text   sentiment
2   @user @user Bloody Sunday was terrible. And ye...  Ambivalent
19  "Ant-Man was ok. Loved the action, first hour ...  Ambivalent
23            Red Sox may be the best worst team ever  Ambivalent
24  "Lovatics loves Demi. some Lovatics bullying o...  Ambivalent
27  Amazing!! We may not have won the 2nd game\u00...  Ambivalent
29  I hope the servers for the new naruto game are...  Ambivalent
35  well bummer the Vikings JUST LOST to WSH :(   ...  Ambivalent
55  Some band nearby playing U2's Sunday Bloody Su...  Ambivalent
62  @user Only if they lose the NFC Championship l...  Ambivalent
64  @user @user  Sarah Palin may be a stupid cow  ...  Ambivalent


In [5]:
ambivalent_df = ambivalent_df[
    ["text", "sentiment"]
].copy()

print(ambivalent_df.columns.tolist())

['text', 'sentiment']


In [7]:
import pandas as pd

# Load the dataset saved from 01_data_exploration.ipynb
train_df = pd.read_csv(
    "../data/raw/tweet_eval_sentiment_train.csv"
)

# Create the original 3-class dataset
original_df = train_df[
    ["text", "sentiment"]
].copy()

print("Original dataset:")
print(original_df.shape)

print("\nClass distribution:")
print(original_df["sentiment"].value_counts())

Original dataset:
(45615, 2)

Class distribution:
sentiment
Neutral     20673
Positive    17849
Negative     7093
Name: count, dtype: int64


In [8]:
final_df = pd.concat(
    [
        original_df,
        ambivalent_df
    ],
    ignore_index=True
)

print("Final dataset shape:", final_df.shape)

Final dataset shape: (45651, 2)


In [9]:
print(final_df["sentiment"].value_counts())

sentiment
Neutral       20673
Positive      17849
Negative       7093
Ambivalent       36
Name: count, dtype: int64


In [10]:
print(
    "Duplicates before:",
    final_df["text"].duplicated().sum()
)

Duplicates before: 65


In [11]:
final_df = final_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

In [12]:
print(
    "Duplicates after:",
    final_df["text"].duplicated().sum()
)

Duplicates after: 0


In [13]:
label_mapping = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2,
    "Ambivalent": 3
}

final_df["label"] = final_df["sentiment"].map(
    label_mapping
)

print(final_df.head())

                                                text sentiment  label
0  "QT @user In the original draft of the 7th boo...  Positive      2
1  "Ben Smith / Smith (concussion) remains out of...   Neutral      1
2  Sorry bout the stream last night I crashed out...   Neutral      1
3  Chase Headley's RBI double in the 8th inning o...   Neutral      1
4  @user Alciato: Bee will invest 150 million in ...  Positive      2


In [14]:
print(
    "Missing labels:",
    final_df["label"].isnull().sum()
)

Missing labels: 0


In [15]:
print(final_df["sentiment"].value_counts())

sentiment
Neutral     20655
Positive    17840
Negative     7091
Name: count, dtype: int64


In [16]:
print(final_df["label"].value_counts().sort_index())

label
0     7091
1    20655
2    17840
Name: count, dtype: int64


In [17]:
final_df.to_csv(
    "../data/processed/final_sentiment_dataset.csv",
    index=False
)

print("Final 4-class dataset saved successfully!")

Final 4-class dataset saved successfully!


In [18]:
print(final_df["sentiment"].value_counts())

sentiment
Neutral     20655
Positive    17840
Negative     7091
Name: count, dtype: int64
